# 9장 — 임베딩은 어떻게 배우나: 대조 학습 🧲

"짝은 가깝게, 남은 멀게"를 **진짜 손실(InfoNCE) + 진짜 경사하강**으로 돌려 봅니다 —
2차원 toy라서 눈으로 전 과정을 볼 수 있어요.

**이 노트북의 목표**
1. InfoNCE 손실을 정의하고, 경사하강(2부 9장)으로 좌표를 직접 학습
2. 학습 후 "각 질문의 최근접 문서 = 정답"이 되는 것(검색이 작동!)을 확인
3. −log 손실 가족(5부 4장·6부 4장)의 세 번째 얼굴 확인

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch9')

## 1. 실험 ① — InfoNCE + 경사하강으로 지도 그리기

질문 3개(Q1~Q3)와 정답 문서 3개(D1~D3). 처음엔 짝이 멀리 흩어져 있습니다.
손실: 각 질문이 소프트맥스로 **정답 문서를 고를 확률**에 −log.

> 처음이어도 괜찮아요 — **소프트맥스**(읽기: 소프트맥스)는 점수 목록을 **합이 1인 확률**로
> 바꾸는 변환(각 점수에 e^s 를 취해 전체 합으로 나눔 — 큰 점수일수록 큰 확률),
> **−log** 는 정답에 준 확률이 1이면 0, 작을수록 치솟는 벌점 변환,
> **경사하강**은 손실이 줄어드는 방향으로 좌표를 조금씩 미는 학습 절차(2부 9장)입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 좌표 파라미터 — Q 3개 + D 3개 (짝: Qi ↔ Di). 처음엔 짝이 멀다.
Q0 = np.array([[2.0, 7.5], [7.5, 7.5], [5.0, 8.2]])
D0 = np.array([[8.0, 2.0], [1.8, 2.2], [5.0, 1.2]])
TAU = 0.7   # 온도 — 유사도 차이를 얼마나 날카롭게 반영할지 (4부 12장의 그 온도)

def info_nce(Q, D):
    """InfoNCE — 각 질문 i가 (거리 기반 소프트맥스로) 정답 Di를 고를 확률에 −log₂."""
    losses = []
    for i in range(len(Q)):
        sims = -np.linalg.norm(D - Q[i], axis=1) / TAU    # 유사도 = −거리 ÷ 온도
        p = np.exp(sims - sims.max())
        p = p / p.sum()                                   # 소프트맥스
        losses.append(-np.log2(p[i]))                     # 정답에 −log (5부 4장의 얼굴)
    return float(np.mean(losses))

def grad_step(Q, D, lr=0.4, eps=1e-4):
    """유한차분으로 기울기를 재서 한 걸음 — 2부 9장의 경사하강 그대로."""
    params = np.concatenate([Q.ravel(), D.ravel()])
    grad = np.zeros_like(params)
    f = lambda v: info_nce(v[:6].reshape(3, 2), v[6:].reshape(3, 2))
    for k in range(len(params)):
        plus = params.copy(); plus[k] += eps
        minus = params.copy(); minus[k] -= eps
        grad[k] = (f(plus) - f(minus)) / (2 * eps)
    params -= lr * grad
    return params[:6].reshape(3, 2), params[6:].reshape(3, 2)

Q, D = Q0.copy(), D0.copy()
history = [info_nce(Q, D)]
for step in range(200):
    Q, D = grad_step(Q, D)
    history.append(info_nce(Q, D))

print(f'손실: {history[0]:.2f}비트 → {history[-1]:.3f}비트')
assert history[-1] < 0.3, '경사하강이 손실을 바닥까지 줄여야'

# 학습 결과: 각 질문의 최근접 문서가 정답인가? (검색이 작동하는 지도가 됐는가)
for i in range(3):
    nearest = int(np.argmin(np.linalg.norm(D - Q[i], axis=1)))
    print(f'  Q{i+1}의 최근접 문서: D{nearest+1}  (정답 D{i+1})')
    assert nearest == i, '학습 후에는 각 질문의 최근접 문서가 정답이어야'
print('\n✔ 확인: "짝은 가깝게, 남은 멀게"만 시켰는데 검색이 되는 지도가 태어났다')

## 2. 학습 전후의 지도 — 구조는 부산물

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (q, d), title in zip(axes, [(Q0, D0), (Q, D)], ['학습 전 (짝이 흩어짐)', '학습 후 (짝은 붙고 남은 멀리)']):
    for i, color in enumerate(['tab:blue', 'tab:orange', 'tab:green']):
        ax.scatter(*q[i], marker='*', s=180, c=color)
        ax.scatter(*d[i], marker='o', s=80, c=color)
        ax.plot([q[i][0], d[i][0]], [q[i][1], d[i][1]], '--', c=color, alpha=0.4)
        ax.annotate(f'Q{i+1}', q[i], textcoords='offset points', xytext=(6, 6), fontsize=9)
        ax.annotate(f'D{i+1}', d[i], textcoords='offset points', xytext=(6, -12), fontsize=9)
    ax.set_title(title); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

pair_d = np.mean([np.linalg.norm(Q[i] - D[i]) for i in range(3)])
non_d = np.mean([np.linalg.norm(Q[i] - D[j]) for i in range(3) for j in range(3) if i != j])
print(f'짝 평균 거리 {pair_d:.2f}  vs  남 평균 거리 {non_d:.2f}')
assert non_d > pair_d * 1.5, '짝은 상대적으로 붙고 남은 떨어져야 — 클러스터는 부산물'
# 흥미로운 관찰: 소프트맥스는 "상대" 비교라 짝이 0까지 붙진 않는다 — 남보다 확실히 가까우면 손실은 이미 바닥.

## 3. 실험 ② — 또 너였구나, −log: 세 번째 재회

5부 4장(다음 토큰), 6부 4장(보상 모델), 그리고 오늘(문서 고르기) — 같은 채점 철학입니다.

In [ ]:
print(' P(정답을 고를 확률)   손실 = −log₂P')
for p, note in [(1/8, '8지선다 찍기'), (1/4, '4지선다 찍기'), (1/2, '반반'), (0.99, '거의 확신')]:
    print(f'   {p:5.2f} ({note:8s})    {-np.log2(p):5.2f}비트')

assert np.isclose(-np.log2(1/4), 2), '구분 못 하면(1/4) 2비트'
assert np.isclose(-np.log2(1/2), 1), '반반이면 1비트'
assert -np.log2(0.99) < 0.02, '확신하고 맞히면 벌점 거의 0'
print('\n✔ 확인: 확신하고 맞히면 0, 헷갈리면 폭발 — 말하기(5부)·취향(6부)·지도(7부)가 같은 수학 위에')

## 4. 관찰 정리 + 🤖 AI 연결

- **대조 학습**: 양성 쌍(질문-정답 문서 등 — 세상에 널림)은 끌고, 음성 쌍(배치 안 이웃 — 공짜)은
  민다. 손실은 **InfoNCE** = 소프트맥스 + 정답에 −log, 옮기는 손은 경사하강(2부 9장).
- 클러스터·의미의 방향(3장)·검색이 되는 지도 — 전부 이 학습의 **부산물**.
- 절대 점수가 아니라 **정답과 오답의 간격**이 손실을 움직인다 — 하드 네거티브(헷갈리는 오답)가
  좋은 선생인 이유. 사내 용어에 약하면 자사 쌍 데이터로 **도메인 파인튜닝**(임베딩판 조련).

**다음 챕터**: 10장 — 🎯 캡스톤. 임베딩 → 코사인 → top-k → 그라운딩,
배운 전부로 **사내 FAQ 검색봇**을 처음부터 끝까지 조립합니다.